# MODEL 3: Without SMOTE and Threshold Tuning

# Purpose & Expected Observation

*   **Purpose**: Evaluates the effect of combining pre-decision features with post-optimization outcome variables to investigate target leakage effects.
*   **Expected Observation**: Substantial increase in Accuracy, ROC-AUC, Recall, and F1-score expected because performance metrics implicitly encode optimization success information. Demonstrates leakage-driven performance inflation.


In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception:
    xgb_available = False
try:
    from lightgbm import LGBMClassifier
    lgbm_available = True
except Exception:
    lgbm_available = False


from pandas.api.types import (
    is_integer_dtype, is_bool_dtype, is_object_dtype
)
from pandas import CategoricalDtype

RANDOM_STATE = 42

# Config
DATA_PATH = "6G_IoT_Beamforming_Dataset.csv"
TEST_SIZE = 0.2 # Added TEST_SIZE

# Load and basic cleaning
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Define X and y earlier
X = df.drop(columns=['Optimized'])
y = df['Optimized']

# Removed performance_features and performance_feature_group as per user's request to not separate features.

target = 'Optimized'  # binary label 0/1

# Define all_feature_groups after df and target are available
# Now, all_feature_groups will only contain a single group with all features (except target)
all_features_list = [col for col in df.columns if col != target]
all_feature_groups = {
    "All Features": all_features_list
}

missing_cols = [c for c in all_feature_groups["All Features"] + [target] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in CSV: {missing_cols}")

# Target to binary
y_raw = df[target]
y = pd.to_numeric(y_raw, errors='coerce')
if y.isna().any():
    y_map = (
        y_raw.astype(str).str.strip().str.lower()
        .map({'1': 1, '0': 0, 'yes': 1, 'no': 0, 'true': 1, 'false': 0,
              'optimized': 1, 'not optimized': 0, 'opt': 1, 'non-opt': 0})
    )
    if y_map.isna().any():
        raise ValueError("Target column cannot be coerced to binary 0/1. Please clean the target values.")
    y = y_map
y = y.astype(int)
if set(y.unique()) - {0, 1}:
    raise ValueError(f"Target must be binary 0/1. Found: {set(y.unique())}")

pos_rate = y.mean()
print(f"Class balance: positive rate = {pos_rate:.3f} ({y.sum()} positives out of {len(y)})")

# Column type detection
def detect_column_types(X_df, explicit_binary=None, explicit_categorical=None):
    explicit_binary = set(explicit_binary or [])
    explicit_categorical = set(explicit_categorical or [])

    binary_cols, categorical_cols, numeric_cols = [], [], []

    for col in X_df.columns:
        s = X_df[col]
        if col in explicit_binary:
            binary_cols.append(col); continue
        if col in explicit_categorical:
            categorical_cols.append(col); continue

        if is_bool_dtype(s):
            binary_cols.append(col)
        elif is_object_dtype(s) or isinstance(s.dtype, CategoricalDtype):
            categorical_cols.append(col)
        else:
            if is_integer_dtype(s):
                uniq = pd.unique(s.dropna())
                if set(uniq).issubset({0, 1}):
                    binary_cols.append(col)
                else:
                    numeric_cols.append(col)
            else:
                uniq = pd.unique(s.dropna())
                if set(uniq).issubset({0.0, 1.0}):
                    binary_cols.append(col)
                else:
                    numeric_cols.append(col)
    return numeric_cols, categorical_cols, binary_cols
explicit_binary = ['Environment_Outdoor', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

# Preprocessing builders
# OneHotEncoder dense for compatibility with SMOTE - SMOTE removed
try:
    categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('to_object', FunctionTransformer(lambda X: np.asarray(X).astype(object))),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', categorical_encoder)
])

binary_transformer = Pipeline(steps=[
    ('to_float', FunctionTransformer(lambda X: np.asarray(X).astype(float))),
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

def build_preprocessor(X_subset):
    num_cols, cat_cols, bin_cols = detect_column_types(X_subset, explicit_binary=explicit_binary)
    preproc = ColumnTransformer(transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols),
        ('bin', binary_transformer, bin_cols)
    ], remainder='drop')
    return preproc

# Model zoo
def get_models(scale_pos_weight=1.0):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(
            n_estimators=400, random_state=RANDOM_STATE, class_weight='balanced_subsample', n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "AdaBoost": AdaBoostClassifier(n_estimators=400, learning_rate=0.05, random_state=RANDOM_STATE),
        "SVM (RBF)": SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
        "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000,
                                     early_stopping=True, n_iter_no_change=10,
                                     random_state=RANDOM_STATE),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        "KNeighbors": KNeighborsClassifier(n_neighbors=5),
        "Extra Trees": ExtraTreesClassifier(n_estimators=400, random_state=RANDOM_STATE, class_weight='balanced_subsample', n_jobs=-1),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE)
    }
    if xgb_available:
        models["XGBoost"] = XGBClassifier(
            random_state=RANDOM_STATE, n_estimators=600, learning_rate=0.05,
            max_depth=4, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.0, reg_lambda=1.0, n_jobs=-1,
            eval_metric='logloss', scale_pos_weight=scale_pos_weight
        )
    if lgbm_available:
        models["LightGBM"] = LGBMClassifier(
            random_state=RANDOM_STATE, n_estimators=600, learning_rate=0.05,
            num_leaves=31, max_depth=-1, colsample_bytree=0.8, subsample=0.8,
            reg_alpha=0.0, reg_lambda=1.0, n_jobs=-1,
            scale_pos_weight=scale_pos_weight
        )
    return models



# Evaluate function per group
def evaluate_group(df, features, y, group_name):
    X = df[features].copy()

    preproc = build_preprocessor(X)

    pos_rate = y.mean()
    neg_rate = 1 - pos_rate
    scale_pos_weight = (neg_rate / pos_rate) if 0 < pos_rate < 1 else 1.0

    models = get_models(scale_pos_weight=scale_pos_weight)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    results = [] # Initialize results list

    for name, model in models.items(): # Added this loop
        # Build the pipeline without SMOTE (removed ImbPipeline/SMOTE)
        pipe_base = Pipeline(steps=[('preprocessor', preproc),
                                    ('classifier', model)])

        # Fit on full training and evaluate
        pipe_base.fit(X_train, y_train)

        # Predictions at default 0.5 threshold (removed threshold tuning)
        y_pred = pipe_base.predict(X_test) # Changed from y_pred_05 to y_pred for clarity

        # Probas for AUC metrics
        if hasattr(pipe_base.named_steps['classifier'], "predict_proba"):
            y_proba_test = pipe_base.predict_proba(X_test)[:, 1]
        elif hasattr(pipe_base.named_steps['classifier'], "decision_function"): # Changed 'pipe' to 'pipe_base'
            scores = pipe_base.named_steps['classifier'].decision_function(X_test)
            y_proba_test = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            y_proba_test = np.zeros_like(y_test, dtype=float)


        # Metrics
        def safe_metrics(y_true, y_pred_values, y_proba_values): # Renamed y_pred, y_proba to avoid confusion with outer scope
            acc = accuracy_score(y_true, y_pred_values)
            prec = precision_score(y_true, y_pred_values, zero_division=0)
            rec = recall_score(y_true, y_pred_values, zero_division=0)
            f1 = f1_score(y_true, y_pred_values, zero_division=0)
            roc = roc_auc_score(y_true, y_proba_values) if y_proba_values is not None and len(np.unique(y_true)) == 2 else np.nan
            pr  = average_precision_score(y_true, y_proba_values) if y_proba_values is not None and len(np.unique(y_true)) == 2 else np.nan
            return acc, prec, rec, f1, roc, pr

        # Using predictions at 0.5 threshold, removed other threshold logic
        acc, prec, rec, f1, roc, prauc = safe_metrics(y_test, y_pred, y_proba_test)


        results.append({
            "Feature_Group": group_name,
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "ROC-AUC": roc,
            "PR-AUC": prauc
        })


    return pd.DataFrame(results)

# Run evaluations
all_results = []
for group_name, features in all_feature_groups.items():
    # The check for empty features will still work, though with the new all_feature_groups it should always be non-empty
    if not features:
        print(f"Skipping evaluation for '{group_name}' as no features are defined.")
        continue
    res = evaluate_group(df, features, y, group_name)
    all_results.append(res)


final_results = pd.concat(all_results, ignore_index=True)

# Sort for readability: by Feature_Group then F1 (removed F1@Tuned)
final_results = final_results.sort_values(by=["Feature_Group", "F1"], ascending=[True, False]) # Changed F1@Tuned to F1

print("\n=== Beamforming Optimization Classification Results ===") # Updated message
print(final_results)

final_results.to_csv("Beamforming_Classification_Results_Improved.csv", index=False)

Class balance: positive rate = 0.168 (168 positives out of 1000)
[LightGBM] [Info] Number of positive: 134, number of negative: 666
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000363 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3076
[LightGBM] [Info] Number of data points in the train set: 800, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.167500 -> initscore=-1.603450
[LightGBM] [Info] Start training from score -1.603450
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
